# Solving Linear Equations

## Systems of Equations
Let's say we have a system of coupled equations, for example, take this system of three equations and three unknowns:

\begin{alignat*}{4}
 x &+ y    &+ z  &= 6 \\
-x &+ 2y   &     &= 3 \\
2x &+      &+ z  &= 5
\end{alignat*}

To solve you could:

1. Use the first equation to eliminate $x$ in the second two equations, resulting in:

   \begin{alignat*}{4}
    x &+ y    &+ z  &= 6 \\
      &+ 3y   &+ z  &= 9 \\
      &- 2y   &- z  &= -7
   \end{alignat*}

2. Use the second equation to eliminate $y$ in the final equation, resulting in:

   \begin{alignat*}{4}
    x &+ y    &+ z  &= 6 \\
      &+ 3y   &+ z  &= 9 \\
      &       &- \tfrac{1}{3}z  &= -1
   \end{alignat*}

   At this point, you would have completed  _forward elimination_. the last
   equation has a single unknown, $z$, and once we solve for it, we
   can substitute the value of $z$ in the equation above, leaving it
   with a single unknown, $y$, and so forth.

3. Now we do _back substitution_.

   From the last equation, we see $z = 3$. Putting this into the
   equation above, we get $y = 2$.  Putting both of these into the
   first equation, we finally find $x = 1$.

This process is the basis of the process of  _Gaussian elimination_.

## Linear Algebra
Of course, there is a whole branch of mathematics built to handle the solution of such systems, in which we represent systems of linear equations in matrix form to solve the equation:
$$
{\bf A}{\bf x} = {\bf v}
$$

In which the solution for the vector of unknown variables is determined by taking the inverse of the coefficient matrix:

$${\bf x} = {\bf A}^{-1} {\bf v}$$


```{warning}
It turns out numerically, actually computing the inverse of a matrix turns out to be complicated and inefficient! 

(In fact, there are actually quite a few things from linear algebra that we don't translate well numerically.)

See: [Seven sins of numerical linear algebra](https://nhigham.com/2022/10/11/seven-sins-of-numerical-linear-algebra/)
```

### Matrix Gaussian Elimination
Numerically, we can instead apply what we know from solving systems of equations above:
1. Multiplying any equation by a constant produces an equivalent equation $\rightarrow$ We can multiply any row of ${\bf A}$ by a constant as long as we also multiply the corresponding row of ${\bf v}$ by the same constant.
2. Any linear combination of two equations produces a valid equation. $\rightarrow$ We can transform any row of the matrix ${\bf A}$ and corresponding row of vector $\bf{v}$ by subtracting a multiple of any other row. 

These two principles allow us to perform the **Gaussian Elimination** in matrix form. 

For example, we can reproduce the first set of equations by writing the coefficients of our equations in matrix ${\bf A}$:
$$
{\bf A} = \left ( \begin{array}{ccc}
                     1  &  1  &  1 \\
                    -1  &  2  &  0 \\
                     2  &  0  &  1 \end{array} \right )
$$

and the RHS of the equations as the vector ${\bf v}$:

$$
{\bf v} = \left ( \begin{array}{c} 6 \\ 3 \\ 5 \end{array} \right )
$$

and we are solving for the vector ${\bf x}$:
$$
{\bf x} = \left ( \begin{array}{c} x \\ y \\ z \end{array} \right )
$$

```{note} [Augmented Matrix Notation](https://en.wikipedia.org/wiki/Augmented_matrix)
For ease of performing the same operations on both sides of our equations, let's represent these equations as an augmented matrix:

$$
({\bf A}|{\bf v}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                    -1  &  2  &  0 & 3 \\
                     2  &  0  &  1 & 5 \end{array} \right )
$$
```

Programmatically, we can use a `numpy` array to set up the matrix and vector:

In [25]:
import numpy as np

A = np.array([[1, 1, 1], [-1, 2,0],[2,0, 1]],dtype=float)
v = np.array([6,3,5],dtype=float)

#### Forward Elimination
The goal of forward elimination is to make ${\bf A}$ into a  [triangular matrix](https://en.wikipedia.org/wiki/Triangular_matrix), you may be familiar with this procedure as getting an augmented matrix into **row echelon form**.

1. When previously we went to eliminate $x$ in the 2nd and 3rd equations using the 1st equation, now we zero out the non-diagonal terms in the first column using the first row. 

This would mean taking the first term of the second row  $A_{21}$, finding the ratio of it relative to the first term of the first row $A_{11}$, mutiplying the first row by that ratio $A_{21}/A_{11}$ and subtracting it from the entire second row. 

Then doing the same thing for the third row: multiply the first row by $A_{31}/A_{11}$ and subract it from the third row of ${\bf A}|{\bf v}$. 

Which leaves us with:

$$
({\bf A}|{\bf v}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                     0  &  3  &  1 & 9 \\
                     0  & -2  & -1 & -7 \end{array} \right )
$$


2. Then, where we continued onward, using the 2nd equation to eliminate $y$ in the 1st and 3rd equations, we now continue this process of row reducing.

Again, now we are scaling the 2nd row by the ratio of second term $A_{i,2}$ in the other rows to that of the diagonal element $A_{22}$ and subtracting off the re-scaled 2nd row from the third row, to get:

$$
({\bf A}|{\bf v}) = \left ( \begin{array}{ccc|c}
                     1  &  1  &  1 & 6 \\
                     0  &  3  &  1 & 9 \\
                     0  &  0  &  -\tfrac{1}{3} & -1 \end{array} \right )
$$


When iterating through arrays, `numpy` arrays iterate through rows first:

In [19]:
for row in A:
    print(row)

print('First row:', A[0,:])
print('First column:', A[:,0])

[1. 1. 1.]
[-1.  2.  0.]
[2. 0. 1.]
First row: [1. 1. 1.]
First column: [ 1. -1.  2.]


So we can use this to our advantage and set up an algorithmic implementation for row-reduction.  

:question
```{embed} ex_5-1
```



In [29]:
N = np.shape(A)[0] # number of rows
for row, k  in zip(A,range(N)):
    a_kk = row[k] #kth term of row 
    v_k = v[k]
    for j in range(k+1,N):
        A_j = A[j,:] #next row
        f_j = A_j[k]/a_kk # coefficient for kth term of row
        A_j += -f_j*row # correct subsequent rows
        v[j]+= -f_j*v_k # adjust RHS of eqn. same way

print("A:" ,A) # note we have changed A and v in place
print("v:", v)


A: [[ 1.          1.          1.        ]
 [ 0.          3.          1.        ]
 [ 0.          0.         -0.33333333]]
v: [ 6.  9. -1.]


#### Back Substitution
Once the matrix is upper triangular, the last row of ${\bf A}$ has only a single non-zero term.

Then we know we know we can commence backsubstitution by solving for the last term of ${\bf x}$ by taking the ratio of the last two entries in the augmented matrix:
$$x_{3} = v_{3}/A_{33}$$

Then we can go up to the previous row, evaluating the linear combination we know that: $ A_{22} x_2 + A_{23} x_3 = v_2$
So then the next term in ${\bf x}$ is then:
$$x_{2} = \frac{v_2 - A_{23} x_3}{A_{22}} = \frac{v_2 - \frac{A_{23}}{A_{33}} v_3}{A_{22}}$$

We can continue recursively until, as before, we end up with:

$${\bf x} = [1,2,3]$$


As a recursive process, we can once again represent this process algorithmically!
We can always solve for the last element:
$$x_N = \frac{v_N}{a_{N,N}}$$ 
So every subsequent element has the form:
$$x_k = \frac{v_k - \sum_{j=k+1}^N a_{k,j} x_j}{a_{k,k}}$$

:question

```{embed} ex_5-2
```

In [30]:
# 
x = np.zeros_like(v)

#
x[N-1] = v[N-1] / A[N-1, N-1]

# 
for k in reversed(range(N-1)):
    vsum = v[k]
    for j in range(k+1, N):
        vsum += -A[k, j] * x[j]
    x[k] = vsum / A[k, k]

print(x)

[1. 2. 3.]


<h2><span class="fa fa-flash"></span> In-Class Coding Exercise </h2>

```{embed} icc_5-1
```

### Pivoting
Now that we have a function that works for a given matrix ${\bf A}$ and vector ${\bf v}$, we should be able to use it on anything!

This would be great, but first we need to figure out if we can actually use it on anything. When designing algorithms, we also need to consider under what situations the code might break. 

Consider the following system:

$$
({\bf A}|{\bf v}) = \left ( \begin{array}{cccc|c}
                     0  &  1  &  4 & 1 & -4 \\
                     3  &  4  &  -1 & -1 & 3 \\
                     1  & -4  & 1 & 5 & 9 \\
                     2 & -2  & 1 & 3 & 7  \end{array} \right )
$$

You can see that right away, we will have a problem as the first element of the first row is $0$, so our first coefficient will be undefined! 

:question

```{embed} ex_5-3
```

```{embed} sol_5-3
```




## LU Decomposition

Gaussian elimination is a set of row operations on ${\bf A}$, which works great if you want solve ${\bf A}{\bf x} = {\bf v}$ once. However, there are situations in physics where we are solving equations with set coefficients but an evolving RHS. In this case, the row operations on ${\bf A}$ themselves wouldn't change, so it would be rather inefficient to keep solving for them!

In **LU decomposition**, we take advantage of the fact that we can write a series of row operations as a transformation matrix, that is, the coefficient matrix ${\bf A}$ is actually a product of a lower triangular matrix ${\bf L}$ and an upper triangular matrix that we solve for with gaussian elimination ${\bf U}$:

$$ {\bf A} = {\bf L} {\bf U}$$

such that:

$$ {\bf L} {\bf U} {\bf x} = {\bf v}$$

Thus, if we perform a gaussian elimination *once*, we can get ${\bf U}$ and define a new vector ${\bf y}$:

$$ {\bf U} {\bf x} = {\bf y}$$

which we can use with any ${\bf v}$:

$$ {\bf L} {\bf y} = {\bf v}$$

Since $L$ is a triangular matrix, we can go straight into a backsubstitution to solve for ${\bf y}$ and then another for ${\bf x}$. 

Generally, an LU decomposition is considered by default to be a fairly efficient way to solve linear equations, except in the case of ${\bf A}$ being any type of sparse matrix -- in which the majority of elements are zero to begin with -- such as for [tridiagonal](https://en.wikipedia.org/wiki/Tridiagonal_matrix) or other types of banded matrices, in which Gaussian elimination and backsubstitution will be more efficient. 

Conveniently, `scipy` has a linear algebra module,`linalg`, that can compute the [LU decomposition](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lu.html) of a given matrix. 

Here we can use it to check that `L @ U` does in fact return our original matrix `A`

In [31]:
from scipy.linalg import lu

A = np.array([[2, 1, 4, 1],
              [3, 4,-1,-1],
              [1,-4, 1, 5],
              [2,-2, 1, 3]],dtype=float) 

v = np.array([ -4, 3, 9, 7 ],dtype=float)

L,U = lu(A,permute_l=True)
print(L@U)

[[ 2.  1.  4.  1.]
 [ 3.  4. -1. -1.]
 [ 1. -4.  1.  5.]
 [ 2. -2.  1.  3.]]



```{note}
`numpy` has it's own `linalg` module, which has a function called [`solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) that uses LU decomposition to solve a system of linear equations.
```


## Additional Linear Algebra Methods
Numerical linear algebra is a whole sub-topic on it's own. As you might imagine, different methods might be more suitable for different types of problems or types of matrices. The `numpy.linalg` module has libraries of implementations of other common solution techniques. 

### Computing an Inverse
As discussed, you can also solve a system of linear equations ${\bf A} {\bf x} = {\bf v}$ by computing the inverse of ${\bf A}$:
$$ {\bf x} = {\bf A^{-1}} v$$

Which we can do with `numpy.inv`



In [32]:
from numpy.linalg import inv

A = np.array([[2, 1, 4, 1],
              [3, 4,-1,-1],
              [1,-4, 1, 5],
              [2,-2, 1, 3]],dtype=float) 

v = np.array([ -4, 3, 9, 7 ],dtype=float)

x = inv(A) @ v

print(x)


[ 2. -1. -2.  1.]


### Ill-conditioned Matrices
Solving ${\bf A} {\bf x} = {\bf v}$ requires ${\bf A}$ to be non-singular, i.e. invertable (have a non-zero determinant). Analytically, we would typically consider problems tractable as long as ${\bf A}$ was non-singular. 

Numerically, even non-singular matrices can pose difficulty if they are *ill-conditioned*, that is if they are *almost* singular by virtue of the roundoff error.

The larger the [**condition number**](https://en.wikipedia.org/wiki/Condition_number) of a matrix, the closer it is to being singular. 

The error in the determination of ${\bf x}$ is a function of the condition number $\mathrm{cond}({\bf A})$:  _for every order of magnitude of the condition number, we lose about 1 digit of accuracy in our solution_.

Comparing an exact solution, ${\bf x}^\star$ with the computed solution, ${\bf x}$:
$$\frac{| {\bf x}^\star - {\bf x} |}{| {\bf x}^\star |} \approx \mathrm{cond}({\bf A}) \cdot \epsilon$$

where $\epsilon$ is the machine epsilon.



In [34]:
from numpy.linalg import cond, det


A = np.array([[1, 1, 1], [-1, 2,0],[2,0, 1]],dtype=float)

B = np.array([[11, 10, 14],[12, 11, -13], [14, 13, -66]],dtype=float)

# A and B both have non-zero determinants (technically invertiable)
print(det(A), det(B))

# Condition number of B is 10,000x that of A!
print(cond(A),cond(B))

-1.0 0.9999999999995668
17.669785440600158 111039.44853422188


See: [example from Garry Tee](https://dl.acm.org/doi/pdf/10.1145/1052668.1052672)

### Eigenvalues and Eigenvectors

There are many times in dealing with systems of equations in physics, that what we want is to find eigenvalues and eigenvectors that satisfy:

$${\bf A} {\bf v} = \lambda {\bf v}$$

for a symmetric or Hermitian matrix ${\bf A}$. 

That is for an $N\times N$ matrix ${\bf A}$, there are $N$ eigenvectors. We can construct an $N\times N$ matrix $V$, for which each column is an eigenvector that satisfies:

$$ {\bf A}{\bf V} = {\bf V}{\bf D}$$

for the corresponding eigenvalues $\lambda_i$ comprise the diagonal elements of the matrix ${\bf D}$.

The algorithmic method for solving the eigenproblem is an iterative decomposition:

#### QR decomposition

Here we decompose the matrix ${\bf A}$ into an orthogonal matrix ${\bf Q}$ and an upper-triangular matrix ${\bf R}$:
$${\bf A} = {\bf Q}_1 {\bf R}_1$$
and define a new matrix that is a product of ${\bf R}_1 {\bf Q}_1$ (order matters!) such that this new matrix
$${\bf A_1} = {\bf Q}_1^T {\bf A} {\bf Q}_1$$ 
(see: Newman 6.2 for details)

By the magic of linear algebra, if you iterate $k$ times, eventually the matrix:

$${\bf A}_k = ( {\bf Q}_k^T ... {\bf Q}_1^T) {\bf A} ( {\bf Q}_1 ... {\bf Q}_k)$$

will eventually be a diagonal matrix, in this case *the* diagonal matrix ${\bf D}$ whose elements comprise the eigenvalues we are looking for. 

The gory details for doing a QR decomposition are provided as an algorithm in Newman Ex. 6.8.

```{note} QR decomposition vs. LU decomposition
:class: dropdown
Since you technically have a triangular matrix, you could use the QR decomposition to solve ${\bf A}{\bf x} = {\bf v}$, where:
$$ {\bf A}{\bf x} = {\bf Q}{\bf R} {\bf x}$$
such that multiplying both sides by ${\bf Q}^T$ gives:
$$ {\bf R}{\bf x} = {\bf Q}^T {\bf v} $$
which is then solvable through a backsubstitution. 

However, the iterative QR decomposition process is significantly less efficient than just doing an LU decomposition in the first place, so typically no one does this.

```

The `numpy` implementation for symmetric matrices is `numpy.linalg.eigh` for the eigenvalues and eigenvectors and `numpy.linalg.eigvalsh` for the eigenvalues alone. 